# Yeast Pipelines: LDA+GBM, PCA+GBM, GBM baseline, and LdaBoost

This notebook provides a clean and reproducible comparison of several pipelines on the Yeast dataset. It mirrors the other datasets' pipeline structure while enforcing the same number of cross-validation folds across all models.

Sections:
- Dataset and preprocessing
- Reproducibility setup and constants
- LDA+GBM vs PCA+GBM (5-fold cross-validated)
- Baseline Gradient Boosting (5-fold cross-validated)
- LdaBoost (5-fold cross-validated)
- Brief conclusions


In [1]:
"""
Yeast pipelines comparison notebook (cleaned and reproducible).

- Dataset: Yeast (UCI)
- Preprocessing: label encoding only (target)
- Models/Pipelines:
  1) LDA + Gradient Boosting
  2) PCA + Gradient Boosting
  3) Gradient Boosting baseline
  4) LdaBoost (custom boosting with LDA projections)
- Evaluation: Stratified K-Fold cross-validation accuracy (5-fold across all models)

This notebook is intended as supplementary material.
"""
# Reproducibility
import os
import random
import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Tuning constants (as requested)
N_ESTIMATOR = 100
LEARNING_RATE = 0.1
MAX_DEPTH = 3

# Data and ML utilities
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_validate
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline

# Ensure local package import for LdaBoost
import sys
from pathlib import Path
PROJECT_ROOT = "/Users/giuliodonninelli/Desktop/LdaBoost_analysis"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from LdaBoost import LdaBoost

# Load dataset
DATA_PATH = str(Path(PROJECT_ROOT) / "real_datasets" / "yeast" / "Data" / "yeast.csv")
data = pd.read_csv(DATA_PATH)

y = data["name"]
X = data.drop(columns="name").copy()

# Encode labels to integers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Quick peek of the data
display(data.head())
print(f"Dataset shape: {data.shape}")


,mcg,gvh,alm,mit,erl,pox,vac,nuc,name
0,0.58,0.61,0.47,0.13,0.5,0.0,0.48,0.22,MIT
1,0.43,0.67,0.48,0.27,0.5,0.0,0.53,0.22,MIT
2,0.64,0.62,0.49,0.15,0.5,0.0,0.53,0.22,MIT
3,0.58,0.44,0.57,0.13,0.5,0.0,0.54,0.22,NUC
4,0.42,0.44,0.48,0.54,0.5,0.0,0.48,0.22,MIT


Dataset shape: (1484, 9)


## Dataset and preprocessing

- Source: Yeast dataset (`yeast.csv`)
- Features: 8 continuous attributes
- Target: `name` (cellular localization), label-encoded to integers
- Preprocessing: label encoding only
- CV strategy: 5-fold across all experiments


## LDA+GBM vs PCA+GBM

We compare two pipelines with 5-fold cross-validation and `accuracy` scoring.


In [2]:
from dataclasses import dataclass, field
from typing import Any, Dict

@dataclass
class DuoBoostCV:
    seed: int = RANDOM_SEED
    _models: Dict[str, Any] = field(init=False, repr=False)

    def _build_models(self, n_features: int) -> Dict[str, Any]:
        half_components = max(1, n_features // 2)
        return {
            "PCA + GBM": Pipeline([
                ("pca", PCA(n_components=half_components, random_state=self.seed)),
                ("gb", GradientBoostingClassifier(
                    n_estimators=N_ESTIMATOR,
                    learning_rate=LEARNING_RATE,
                    max_depth=MAX_DEPTH,
                    random_state=self.seed,
                )),
            ]),
            "LDA + GBM": Pipeline([
                ("lda", LDA(n_components=None)),
                ("gb", GradientBoostingClassifier(
                    n_estimators=N_ESTIMATOR,
                    learning_rate=LEARNING_RATE,
                    max_depth=MAX_DEPTH,
                    random_state=self.seed,
                )),
            ]),
        }

    def fit(self, X_in, y_in, **cv_kwargs):
        X_arr = np.asarray(X_in)
        y_arr = np.asarray(y_in)
        cv_kwargs = cv_kwargs.copy()
        cv_kwargs.setdefault("scoring", "accuracy")
        cv_kwargs.setdefault("cv", 10)
        cv_kwargs.setdefault("n_jobs", -1)
        cv_kwargs.setdefault("return_train_score", True)

        self._models = self._build_models(X_arr.shape[1])
        scores: Dict[str, Dict[str, float]] = {}
        for name, model in self._models.items():
            cv_res = cross_validate(model, X_arr, y_arr, **cv_kwargs)
            metric_key = next(k for k in cv_res if k.startswith("test_"))
            metric_name = metric_key.replace("test_", "")
            scores[name] = {
                f"mean_{metric_name}": float(cv_res[metric_key].mean()),
                f"std_{metric_name}": float(cv_res[metric_key].std()),
            }
        self.results_ = pd.DataFrame(scores).T.sort_values(by=f"mean_{metric_name}", ascending=False)
        return self.results_

    def __call__(self, X_in, y_in, **cv_kwargs):
        return self.fit(X_in, y_in, **cv_kwargs)

trainer = DuoBoostCV(seed=RANDOM_SEED)
results_duo = trainer(X, y_encoded, cv=5, scoring="accuracy", n_jobs=-1, return_train_score=True)
display(results_duo.round(4))


,mean_score,std_score
LDA + GBM,0.5694,0.0351
PCA + GBM,0.5041,0.0232


## Baseline Gradient Boosting



In [3]:
gb_clf = GradientBoostingClassifier(
    n_estimators=N_ESTIMATOR,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_SEED,
)

cv_scores = cross_validate(
    gb_clf,
    X,
    y_encoded,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

test_key = next(k for k in cv_scores if k.startswith("test_"))
metric_name = test_key.replace("test_", "")
mean_acc = float(cv_scores[test_key].mean())
std_acc = float(cv_scores[test_key].std())
print(f"GBM {metric_name}: {mean_acc:.4f} ± {std_acc:.4f}")


GBM score: 0.5681 ± 0.0369


## LdaBoost

We evaluate the custom `LdaBoost` classifier imported from the local package, using the same constants and folds.


In [4]:
ldaboost = LdaBoost(
    n_estimators=N_ESTIMATOR,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_SEED,
)

accs = ldaboost.cross_validate(X=np.asarray(X), y=np.asarray(y_encoded), cv=5)
print(f"LdaBoost accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")


LdaBoost accuracy: 0.5890 ± 0.0323
